# Round-0 accuracy and calibration

Round 0 is probed before any neighbour information is shown, so an agent's round-0 belief depends only on
its model, its role and the statement. This notebook produces the three tables of the round-0 appendix:

1. round-0 accuracy and calibration per checkpoint (scenario IV, matched roles);
2. pooled accuracy and top-label probability (confidence) by scenario;
3. scenario contrasts with and without adjustment for round-0 confidence.

The 20 statements were maximin-selected to cover disagreement with the medical LLM and its errors
(Appendix D.3), so round-0 accuracy describes the priors agents bring into the interaction, not checkpoint
capability.

Outputs go to `data/analysis/aggregated_expanded/round0/`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from exp_utils import REPO_ROOT, STANCE_LABEL_TO_IDX
from exp_plot_utils import _configure_fonts, SCENARIO_EXTENDED_MAP, SCENARIO_ORDER

from src.analysis.loaders import load_agents_data, load_runs_metadata

_configure_fonts()

RUNS_DIR = REPO_ROOT / "data" / "outputs" / "runs"
DATA_PATH = REPO_ROOT / "data" / "outputs" / "aggregated_expanded"
OUT_PATH = REPO_ROOT / "data" / "analysis" / "aggregated_expanded" / "round0"
OUT_PATH.mkdir(parents=True, exist_ok=True)

KEY = ["run_scenario", "agent_model", "agent_role", "statement_id"]
# belief_ful class order is [P(true), P(false), P(neither)]
P_COLS = ["p_true", "p_false", "p_neither"]

runs_df = pd.read_parquet(DATA_PATH / "run_level_data.parquet")
agents = pd.read_parquet(DATA_PATH / "agent_level_data.parquet")

## Load round-0 beliefs

In [ ]:
runs = load_runs_metadata(RUNS_DIR)
runs = runs[
    runs["validated"] & runs["completed"] & runs["setting"].isin(SCENARIO_ORDER)
]
print(f"Runs: {len(runs)}")

# The aggregated parquets keep only trajectory summaries, so round-0 beliefs come from the raw runs.
frames = []
for run in runs.itertuples(index=False):
    run_agents = load_agents_data(run.run_path)
    b0 = run_agents["belief_ful"][:, 0, :]  # (N_agents, 3) at round 0
    frames.append(
        pd.DataFrame(
            {
                "run_path": run.run_path,
                "run_scenario": run.setting,
                "statement_id": run.statement_id,
                "agent_model": list(run_agents["models"].values()),
                "agent_role": list(run_agents["roles"].values()),
                **dict(zip(P_COLS, b0.T)),
            }
        )
    )
df0 = pd.concat(frames, ignore_index=True)
print(df0.shape)

## Unit of analysis: the (scenario, model, role, statement) cell

The round-0 prompt has no neighbour summary, so every run of a cell asks the same question. The unit of
analysis is therefore the cell, not the run: counting every run would inflate n by the number of runs per cell.

In [ ]:
cells = df0.groupby(KEY, as_index=False).agg(**{c: (c, "mean") for c in P_COLS})

truth = runs_df.drop_duplicates("statement_id").set_index("statement_id")[
    "statement_ground_truth"
]
cells["statement_ground_truth"] = cells["statement_id"].map(truth)
cells["truth_idx"] = cells["statement_ground_truth"].map(STANCE_LABEL_TO_IDX)
assert cells["truth_idx"].notna().all()
cells["truth_idx"] = cells["truth_idx"].astype(int)
cells["pred_idx"] = cells[P_COLS].to_numpy().argmax(axis=1)
cells["conf0"] = cells[P_COLS].to_numpy().max(axis=1)
cells.to_csv(OUT_PATH / "round0_cells.csv", index=False)
print(cells.groupby("run_scenario").size())

## Scores

All scores treat the 3-way distribution as given; *neither* is never the correct label, so it counts as
wrong for accuracy and Brier. Brier is the squared error averaged over the three options (range 0-2/3;
a uniform guess scores 0.22). *mAP* is the mean of the average precision for the true class (statements
ranked by P(true)) and the false class (ranked by P(false)); it is threshold-free, so it separates
discrimination from the yes-bias that *says true* exposes (chance = 0.50 on the balanced set). Because the statement set is 10 true / 10 false, *says true* (share of
argmax = true) shows whether a checkpoint's accuracy comes from a yes-bias.
Per-checkpoint ECE is not reported: with 20 statements per checkpoint, 10-bin ECE is mostly noise.
Brier score and log loss are proper scoring rules and are reported instead; ECE is pooled across checkpoints.

In [ ]:
from sklearn.metrics import average_precision_score


def round0_scores(g: pd.DataFrame) -> dict:
    P = g[P_COLS].to_numpy(dtype=float)
    y = g["truth_idx"].to_numpy()
    pred = P.argmax(axis=1)
    p_y = P[np.arange(len(y)), y]
    return {
        "n": len(g),
        "accuracy": float((pred == y).mean()),
        "says_true": float((pred == 0).mean()),
        "abstain": float((pred == 2).mean()),
        "p_true_label": float(p_y.mean()),
        "confidence": float(P.max(axis=1).mean()),
        "confidence_sd": float(P.max(axis=1).std(ddof=1)),
        "brier": float(((P - np.eye(3)[y]) ** 2).mean(axis=1).mean()),
        # threshold-free: mean AP of the true class (ranked by P(true)) and the false class (ranked by P(false))
        "mAP": float(
            (
                average_precision_score(y == 0, P[:, 0])
                + average_precision_score(y == 1, P[:, 1])
            )
            / 2
        ),
        "log_loss": float(-np.log(np.clip(p_y, 1e-12, None)).mean()),
    }


def score_table(df: pd.DataFrame, by: list[str]) -> pd.DataFrame:
    rows = []
    for keys, g in df.groupby(by, observed=True):
        rows.append({**dict(zip(by, keys)), **round0_scores(g)})
    return pd.DataFrame(rows)

## Main table: checkpoints under matched roles (scenario IV)

16 rows: 15 checkpoints, with `llama-base` appearing under both of its roles (LLM, Human Participant).
Each row covers the 20 statements.

In [ ]:
hf_paths = {}
for f in (REPO_ROOT / "src" / "configs" / "model").glob("*.yaml"):
    cfg = yaml.safe_load(f.read_text())
    if cfg and cfg.get("name") and cfg.get("hf_path"):
        hf_paths[cfg["name"]] = cfg["hf_path"]

calib = score_table(
    cells[cells["run_scenario"] == "experts"], ["agent_model", "agent_role"]
)
calib.insert(1, "hf_path", calib["agent_model"].map(hf_paths))
calib = calib.sort_values("accuracy", ascending=False).reset_index(drop=True)
calib.to_csv(OUT_PATH / "round0_calibration_experts.csv", index=False)
calib.round(3)

## Pooled accuracy and top-label probability by scenario

In [ ]:
def ece_table(df: pd.DataFrame, n_bins: int = 10) -> tuple[float, pd.DataFrame]:
    P = df[P_COLS].to_numpy(dtype=float)
    conf = P.max(axis=1)
    correct = (P.argmax(axis=1) == df["truth_idx"].to_numpy()).astype(float)
    bins = np.clip((conf * n_bins).astype(int), 0, n_bins - 1)
    tab = (
        pd.DataFrame({"bin": bins, "conf": conf, "correct": correct})
        .groupby("bin")
        .agg(
            n=("conf", "size"),
            confidence=("conf", "mean"),
            accuracy=("correct", "mean"),
        )
        .reset_index()
    )
    ece = float(
        (tab["n"] * (tab["accuracy"] - tab["confidence"]).abs()).sum() / len(df)
    )
    return ece, tab


ece_rows = []
for scenario in SCENARIO_ORDER:
    sub = cells[cells["run_scenario"] == scenario]
    ece, _ = ece_table(sub)
    ece_rows.append(
        {
            "run_scenario": scenario,
            "n_cells": len(sub),
            "ece": ece,
            **round0_scores(sub),
        }
    )
ece_df = pd.DataFrame(ece_rows).set_index("run_scenario")
ece_df.to_csv(OUT_PATH / "round0_ece_pooled.csv")
ece_df.round(3)

## Are the scenario effects a confidence artifact?

Specialists start less confident than the generalist, so they might revise more only because they are
less certain. We refit the E2 agent-level models (plasticity, directedness) and the E3 run-level model
(consensus change) with the same random-effects structures, without and with round-0 confidence as a
centred covariate, and compare the scenario contrasts. Agent level: the agent's round-0 top-label
probability. Run level: the population mean of it. EMMs are evaluated at the mean confidence.

In [ ]:
sys.path.insert(0, str(REPO_ROOT / "notebooks"))
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from r_utils import fit_lmer_with_contrasts

# Same random effects as E2 (agent level) and E3 (run level).
AGENT_RANDOM_EFFECTS = (
    "(1 | statement_id) + (1 | run_scenario:statement_id) + (1 | graph_family:statement_id)"
    " + (1 | graph_id) + (1 | run_scenario:graph_id)"
    " + (1 | run_id) + (1 | agent_id:graph_id:run_scenario)"
)
RUN_RANDOM_EFFECTS = (
    "(1 | statement_id) + (1 | run_scenario:statement_id)"
    " + (1 | graph_id) + (1 | run_scenario:graph_id)"
)
OUTCOME_COLS = ["agent_plasticity", "agent_monotonicity"]
GROUPING_COLS = [
    "graph_family",
    "graph_id",
    "run_id",
    "agent_id",
]  # statement_id is already in KEY

# Only the columns the models use go to R; the merge keys are cast to match `cells` (plain str).
agents_m = (
    agents[KEY + GROUPING_COLS + OUTCOME_COLS]
    .astype({c: str for c in KEY + ["run_id"]})
    .merge(cells[KEY + ["conf0"]], on=KEY, how="left", validate="many_to_one")
)
assert agents_m["conf0"].notna().all()
agents_m["conf0_c"] = agents_m["conf0"] - agents_m["conf0"].mean()

runs_m = (
    runs_df[
        [
            "run_scenario",
            "graph_family",
            "statement_id",
            "graph_id",
            "run_id",
            "pop_consensus_change",
        ]
    ]
    .astype({"run_scenario": str, "run_id": str})
    .merge(
        agents_m.groupby("run_id")["conf0"].mean(),
        on="run_id",
        how="left",
        validate="one_to_one",
    )
)
assert runs_m["conf0"].notna().all()
runs_m["conf0_c"] = runs_m["conf0"] - runs_m["conf0"].mean()

print(agents_m.groupby("run_scenario")["conf0"].mean().round(3))

In [ ]:
OUTCOMES = [
    ("agent_plasticity", agents_m, AGENT_RANDOM_EFFECTS),
    ("agent_monotonicity", agents_m, AGENT_RANDOM_EFFECTS),
    ("pop_consensus_change", runs_m, RUN_RANDOM_EFFECTS),
]
R_COLS = {"t.ratio": "t", "p.value": "p", "lower.CL": "lower", "upper.CL": "upper"}

adj_rows = []
for outcome, data, random_effects in OUTCOMES:
    for adjusted in (False, True):
        res = fit_lmer_with_contrasts(
            data,
            outcome=outcome,
            random_effects=random_effects,
            scenario_levels=SCENARIO_ORDER,
            spec_extra="* graph_family" + (" + conf0_c" if adjusted else ""),
            group_var="graph_family",
        )
        ct = (
            res["contrasts_marginal"]
            .rename(columns=R_COLS)
            .assign(outcome=outcome, adjusted=adjusted)
        )
        if adjusted:
            # fit_lmer_with_contrasts leaves the fitted model in R as `df_model`.
            with (ro.default_converter + pandas2ri.converter).context():
                coef = ro.conversion.rpy2py(
                    ro.r("as.data.frame(coef(summary(df_model)))")
                )
            ct["conf0_slope"] = coef.loc["conf0_c", "Estimate"]
            ct["conf0_slope_p"] = coef.loc["conf0_c", "Pr(>|t|)"]
        adj_rows.append(ct)
        print(f"done: {outcome} adjusted={adjusted}")

conf_adj = pd.concat(adj_rows, ignore_index=True)
wide = conf_adj.pivot_table(
    index=["outcome", "contrast"], columns="adjusted", values="estimate"
)
conf_adj = conf_adj.merge(
    (1 - wide[True] / wide[False]).rename("share_explained_by_conf0").reset_index(),
    on=["outcome", "contrast"],
)
conf_adj.to_csv(OUT_PATH / "round0_confidence_adjusted_contrasts.csv", index=False)
conf_adj[
    [
        "outcome",
        "contrast",
        "adjusted",
        "estimate",
        "SE",
        "df",
        "lower",
        "upper",
        "p",
        "cohens_d_total",
        "conf0_slope",
        "share_explained_by_conf0",
    ]
].round(4)

## LaTeX table for Appendix G

In [ ]:
def tex_escape(s: str) -> str:
    return str(s).replace("_", r"\_").replace("&", r"\&")


lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\small",
    r"\begin{tabular}{llccccccc}",
    r"\toprule",
    r"\textbf{Checkpoint} & \textbf{Role} & \textbf{Accuracy} & \textbf{Says true} & \textbf{Abstain} "
    r"& \textbf{\(\bar p(y)\)} & \textbf{mAP} & \textbf{Brier} & \textbf{Log loss} \\",
    r"\midrule",
]
for r in calib.itertuples(index=False):
    lines.append(
        f"\\texttt{{{tex_escape(r.hf_path)}}} & {tex_escape(r.agent_role)} & "
        f"{r.accuracy:.2f} & {r.says_true:.2f} & {r.abstain:.2f} & "
        f"{r.p_true_label:.2f} & {r.mAP:.2f} & {r.brier:.2f} & {r.log_loss:.2f} \\\\"
    )
ece_iv = ece_df.loc["experts", "ece"]
lines += [
    r"\bottomrule",
    r"\end{tabular}",
    r"\caption{\textbf{Round-0 accuracy and calibration per checkpoint} (scenario IV, matched roles; "
    r"\(n=20\) statements per row). Round~0 is probed before any neighbour "
    r"information, so it reflects each checkpoint's prior. \textit{Says true}: share of statements labelled "
    r"\textit{correct} (the set is 10 true / 10 false); \textit{Abstain}: share labelled \textit{neither}; "
    r"\(\bar p(y)\): mean probability on the DrugBank-derived label; \textit{mAP}: mean average precision of "
    r"the true and false classes (threshold-free, chance \(0.50\)); Brier: squared error averaged over the "
    r"three options (uniform guess \(0.22\)). Per-checkpoint ECE is omitted "
    f"(20 items); pooled across checkpoints, ECE \\(= {ece_iv:.2f}\\). "
    r"Statements were selected to enrich disagreement with the medical LLM (Appendix~\ref{app:maximin}), "
    r"so these values describe the priors agents bring into the interaction, not checkpoint capability.}",
    r"\label{tab:round0-calibration}",
    r"\end{table*}",
]
tex = "\n".join(lines)
(OUT_PATH / "round0_calibration_table.tex").write_text(tex)
print(tex)

In [ ]:
# Pooled calibration by scenario (tab:round0-ece); one row per scenario from ece_df.
lines = [
    r"\begin{table}[t]",
    r"\centering",
    r"\small",
    r"\begin{tabular}{lccccc}",
    r"\toprule",
    r"Scenario & Cells & Accuracy & Top-label prob. (\(\pm\) SD) & mAP & ECE \\",
    r"\midrule",
]
for scenario in SCENARIO_ORDER:
    r = ece_df.loc[scenario]
    lines.append(
        f"{SCENARIO_EXTENDED_MAP[scenario]} & {int(r.n_cells):,} & {r.accuracy:.2f} & "
        f"{r.confidence:.2f} \\(\\pm\\) {r.confidence_sd:.2f} & {r.mAP:.2f} & {r.ece:.2f} \\\\"
    )
lines += [
    r"\bottomrule",
    r"\end{tabular}",
    r"\caption{\textbf{Pooled round-0 calibration by scenario.} A cell is a unique (LLM, role, statement) "
    r"combination; all runs of a cell receive the same round-0 prompt. \emph{Top-label prob.}: mean "
    r"(\(\pm\) SD across cells) probability of the chosen option, i.e., confidence. \emph{mAP}: mean average "
    r"precision of the true and false classes, pooled over all cells of the scenario. ECE: expected calibration "
    r"error with 10 equal-width confidence bins (lower is better). Scenario I has only 20 cells, so its ECE is noisy.}",
    r"\label{tab:round0-ece}",
    r"\end{table}",
]
tex = "\n".join(lines).replace("4,540", "4{,}540")
(OUT_PATH / "round0_ece_table.tex").write_text(tex)
print(tex)
